# Prepare DeepMD datasets with `dpdata`

This notebook converts an ASE-readable trajectory into DeepMD training data.

## What it does
- loads a labelled trajectory containing energies and forces
- inspects the number of configurations
- exports the dataset to DeepMD `raw` and `npy` formats
- optionally writes an XYZ file for quick inspection

## Before you run it
Run this notebook from the DeepMD Python environment.
Place one or more `.traj` files in `data_preparation/input/`, then update the configuration cell below if needed.


In [ ]:
from pathlib import Path

import ase.io
import dpdata


## Configuration

Select the trajectory from `data_preparation/input/` and choose the output directory here.
Relative paths are resolved from the repository root.


In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "data_preparation" else Path.cwd().resolve()

INPUT_DIR = PROJECT_ROOT / "data_preparation" / "input"
TRAJECTORY_PATTERN = "*.traj"
TRAJECTORY_INDEX = 0
OUTPUT_DIR = PROJECT_ROOT / "data_preparation" / "outputs" / "dpdata_export"
WRITE_XYZ = True

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")


In [ ]:
candidate_files = sorted(INPUT_DIR.glob(TRAJECTORY_PATTERN))

if not candidate_files:
    raise FileNotFoundError(
        f"No trajectory file matched '{TRAJECTORY_PATTERN}' in {INPUT_DIR}\n"
        "Add one or more `.traj` files to data_preparation/input/ or update TRAJECTORY_PATTERN."
    )

if TRAJECTORY_INDEX >= len(candidate_files):
    raise IndexError(
        f"TRAJECTORY_INDEX={TRAJECTORY_INDEX} but only {len(candidate_files)} matching file(s) were found."
    )

print("Available trajectory files:")
for idx, path in enumerate(candidate_files):
    print(f"  [{idx}] {path.name}")

TRAJECTORY_FILE = candidate_files[TRAJECTORY_INDEX]
print(f"Selected trajectory: {TRAJECTORY_FILE.name}")

ase_frames = ase.io.read(TRAJECTORY_FILE, index=":")
n_frames = len(ase_frames)

print(f"Loaded {n_frames} configurations from {TRAJECTORY_FILE.name}")


## Convert to `dpdata`

`LabeledSystem` expects energies and forces to be available in the input trajectory.


In [ ]:
labeled_system = dpdata.LabeledSystem(
    str(TRAJECTORY_FILE),
    set_size=n_frames,
    fmt="ase/traj",
)

labeled_system


In [ ]:
raw_dir = OUTPUT_DIR / "raw"
npy_dir = OUTPUT_DIR / "npy"

labeled_system.to("deepmd/raw", str(raw_dir))
labeled_system.to("deepmd/npy", str(npy_dir))

print(f"Wrote DeepMD raw dataset to: {raw_dir}")
print(f"Wrote DeepMD npy dataset to: {npy_dir}")


In [ ]:
if WRITE_XYZ:
    xyz_path = OUTPUT_DIR / "dataset.xyz"
    labeled_system.to("xyz", str(xyz_path))
    print(f"Wrote XYZ preview to: {xyz_path}")
else:
    print("Skipping XYZ export.")


## Notes

Typical next step: use the exported `npy` dataset in your DeepMD training workflow.
